# E6 · Curvas ROC del detector

**Spec:** [`docs/spec_E6_codex_roc_curves.md`](../docs/spec_E6_codex_roc_curves.md)  |  **Bloque:** E · Resultado  |  **Run por defecto:** `ROXs12b_realigned`

ROC (DP vs FAP) del matched filter por método y separación, con inyecciones a contraste tipo límite y nulo empírico de anillos + canales sin línea (Julo+25 Fig. 11).

| | |
|---|---|
| **Entrada** | Cubos residuales C5/C6 + E5 QC (contrast_50) + PSF C1 |
| **Salida (QC/productos)** | `tables/roc_curves.csv`, `stages/stage_h06_qc.json` |
| **Consume aguas abajo** | Insumo informativo del checkpoint D1 v3 (robustez por método) |


## Qué hace E6

Complementa a E5: en vez de fijar el FAP (5σ) y variar el contraste, fija el contraste (el `contrast_50` de E5, régimen de límite de detección) y barre el umbral, midiendo DP con inyecciones en anillo (16 ángulos) y FAP con el nulo EMPÍRICO: píxeles del anillo del mapa z base + mapas z reconstruidos con la plantilla centrada en canales libres de línea (paper §3.3.3 — sin supuestos gaussianos). AUC por método y separación.


## Cómo ejecutar de forma independiente

```bash
conda activate MUSE               # kernel/env con astropy + musepipe
export RUN=ROXs12b_realigned   # o ROXs12b_B_adp para comparar
cd MUSE-accretion-pipeline                    # raíz del repo
python -m musepipe.stages.stage_h06_roc --run-id $RUN
```

Moderado (~min: mapas nulos según h06_null_step_channels).

La celda de abajo hace lo mismo desde el notebook (guardada por `RUN`).


In [ ]:
import os, sys
# Localiza la raíz del repo ascendiendo hasta encontrar `musepipe/` (robusto a
# la profundidad: funciona con el cwd en notebooks/<obj>/, en notebooks/ o en la
# raíz). Añade la raíz (para `import musepipe`) y notebooks/ (para `_nbcommon`).
_d = os.getcwd()
while _d != os.path.dirname(_d):
    if os.path.isdir(os.path.join(_d, 'musepipe')) and os.path.isdir(os.path.join(_d, 'notebooks')):
        break
    _d = os.path.dirname(_d)
_root = _d
for _p in (_root, os.path.join(_root, 'notebooks')):
    if _p not in sys.path:
        sys.path.insert(0, _p)
import _nbcommon as nb
RUN_ID = nb.resolve_run_id('ROXs12b_realigned')
print('run  =', RUN_ID)
print('root =', _root)
print('dir  =', nb.run_dir(RUN_ID))
print('QC   =', nb.provenance_line('stages/stage_h06_qc.json', RUN_ID))


## Ejecutar o auditar


In [ ]:
RUN = False   # -> True para RE-EJECUTAR esta etapa (regenera su QC)

if RUN:
    cmd = 'python -m musepipe.stages.stage_h06_roc --run-id $RUN'.replace('$RUN', RUN_ID)
    print('ejecutando:', cmd)
    import subprocess
    subprocess.run(cmd, shell=True, cwd=str(nb.project_root()), check=True)
else:
    print('Modo auditoría (RUN=False): se carga el QC existente abajo.')


## QC / resultados


In [ ]:
qc = nb.load_qc_optional('stages/stage_h06_qc.json', RUN_ID)
nb.show(qc, keys=['params.n_null_wavelengths', 'checks.v1_curves_written', 'checks.v2_auc_above_random', 'checks.v3_null_sample_ok'], title='E6')


## Evidencia: AUC por método y escenario


In [ ]:
if qc is None:
    print('(evidencia omitida: la etapa no se ha ejecutado para esta cadena)')
else:
    with nb.evidence_guard('E6', 'stages/stage_h06_qc.json'):
        q = nb.load_qc('stages/stage_h06_qc.json', RUN_ID)
        print('mapas nulos:', q['params']['n_null_wavelengths'])
        for m, mq in q['methods'].items():
            for s in mq['scenarios']:
                print(f"  {m:5s} r={s['separation_px']:5.1f}px c={s['contrast']:.2e} "
                      f"AUC={s['auc']:.3f} (n_iny={s['n_injections']}, n_nulo={s['n_null']})")
        print('checks:', q['checks'])


## Plot — ROC por método y separación (Fig. 11 del paper)


In [ ]:
try:
    import csv
    import matplotlib.pyplot as plt
    rd = nb.run_dir(RUN_ID)
    rows = list(csv.DictReader(open(rd / 'tables' / 'roc_curves.csv')))
    fig, ax = plt.subplots(figsize=(6.4, 5.4))
    keys = sorted({(r['method'], r['separation_px']) for r in rows})
    for m, sep in keys:
        pts = sorted((float(r['fap']), float(r['dp'])) for r in rows
                     if r['method'] == m and r['separation_px'] == sep)
        ax.plot([p[0] for p in pts], [p[1] for p in pts], lw=1.1, label=f'{m} r={float(sep):g}px')
    ax.plot([0, 1], [0, 1], 'k:', lw=0.8, label='aleatorio')
    ax.set_xlabel('FAP'); ax.set_ylabel('DP'); ax.legend(fontsize=7)
    ax.set_title('E6 · ROC'); fig.tight_layout(); plt.show()
except Exception as e:
    print('No se pudo generar el plot:', type(e).__name__, e)


## Decisiones y notas
- Nulo empírico (anillos + canales sin línea), nunca gaussiano asumido; informativo para D1 v3, no cambia veredictos. · [`docs/spec_E6_codex_roc_curves.md`](../docs/spec_E6_codex_roc_curves.md)


## Checks


In [ ]:
try:
    q = nb.load_qc('stages/stage_h06_qc.json', RUN_ID)
    for k, v in q['checks'].items():
        print(f'  {k}: {v}')
except FileNotFoundError as e:
    print('QC aún no existe para este run:', e)


## Estado

**Pendiente de primera ejecución sobre datos reales** (requiere C5/C6 y E5). Núcleo y contrato verificados con tests sintéticos (2026-07-14/15).
